In [15]:
import os

In [16]:
%pwd

'c:\\Users\\km931\\OneDrive\\Desktop\\BTP project\\Kidney-Disease-Classification-Deep-Learning-Project'

In [4]:
os.chdir("../")

In [17]:
%pwd

'c:\\Users\\km931\\OneDrive\\Desktop\\BTP project\\Kidney-Disease-Classification-Deep-Learning-Project'

In [18]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/kajalmishra0829/Kidney-Disease-Classification-Deep-Learning-Project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="kajalmishra0829"
os.environ["MLFLOW_TRACKING_PASSWORD"]="f37d1ac942a96264ad42ad8cb10ceff43222e048"

In [19]:
import tensorflow as tf

In [20]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [23]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [25]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/kajalmishra0829/Kidney-Disease-Classification-Deep-Learning-Project.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [26]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [27]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            # Log parameters
            mlflow.log_params(self.config.all_params)
    
            # Log metrics
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })
    
            # Save lightweight model
            self.model.save("model.h5")
    
            # Upload model artifact
            mlflow.log_artifact("model.h5")
    
            print("MLflow logging completed successfully")


In [28]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-05-19 17:24:46,997: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-19 17:24:47,006: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-19 17:24:47,013: INFO: common: created directory at: artifacts]
Found 139 images belonging to 2 classes.
9/9 [==============================] - 15s 2s/step - loss: 0.0520 - accuracy: 1.0000
[2026-05-19 17:25:02,631: INFO: common: json file saved at: scores.json]
MLflow logging completed successfully
